# Challenge Alura / OCI — BuildTech AI Agent (Constructora Nova Build SpA)

Este notebook contiene el prototipo completo local para el **Challenge Alura / OCI** de la **Constructora Nova Build SpA**.

### Módulos principales:
1. **Generación automática del Dataset:** Archivos PDF de catálogo/proyectos y CSVs de inventario y personal.
2. **Herramienta 1 (PDF / RAG):** Carga del PDF, chunking, generación de embeddings con Gemini y almacenamiento vectorial en ChromaDB.
3. **Herramientas 2 y 3 (CSV / Structured Query):** Consultas sobre inventario y nómina de trabajadores.
4. **Agente Orquestador (ReAct):** Unifica las herramientas en un agente inteligente capaz de enrutamiento automático.

## Paso 1: Instalación de Dependencias
Ejecuta la siguiente celda para instalar LangChain, Google GenAI SDK, ChromaDB, WeasyPrint, PyPDF y Pandas.

In [ ]:
# Instalación de paquetes necesarios
!pip install -q -U langchain langchain-google-genai langchain-community chromadb pypdf pandas weasyprint

## Paso 2: Generación del Dataset (PDF y CSVs)
Generamos los datos estructurados y no estructurados necesarios para la prueba del agente.

In [ ]:
import os
import pandas as pd
from weasyprint import HTML

# Crear carpeta para los datos
os.makedirs("constructora_data", exist_ok=True)

# 1. Generar CSV Inventario
inventario_data = [
    {
        "id_material": "MAT-001",
        "material": "Cemento Melón Especial 25kg",
        "categoria": "Obra Gruesa",
        "stock_actual": 450,
        "unidad": "Sacos",
        "ubicacion_almacen": "Bodega Central - Galpón A",
        "distribuidor_principal": "Distribuidora El Teniente",
        "precio_unitario_clp": 4800,
        "precio_al_por_mayor_clp": 4200,
        "minimo_por_mayor": 100,
        "estado_stock": "Disponible"
    },
    {
        "id_material": "MAT-002",
        "material": "Fierro Estructural A63-42H 12mm x 12m",
        "categoria": "Acero y Estructuras",
        "stock_actual": 85,
        "unidad": "Barras",
        "ubicacion_almacen": "Bodega Central - Patio Exterior",
        "distribuidor_principal": "AceroNorte Chile",
        "precio_unitario_clp": 14500,
        "precio_al_por_mayor_clp": 12800,
        "minimo_por_mayor": 50,
        "estado_stock": "Stock Crítico"
    },
    {
        "id_material": "MAT-003",
        "material": "Hormigón Preparado H30",
        "categoria": "Obra Gruesa",
        "stock_actual": 0,
        "unidad": "m3",
        "ubicacion_almacen": "Despacho Directo en Obra",
        "distribuidor_principal": "ReadyMix Valparaíso",
        "precio_unitario_clp": 78000,
        "precio_al_por_mayor_clp": 71000,
        "minimo_por_mayor": 20,
        "estado_stock": "Bajo Pedido"
    },
    {
        "id_material": "MAT-004",
        "material": "Ladrillo Princesa 29x14x7.1 cm",
        "categoria": "Albañilería",
        "stock_actual": 3200,
        "unidad": "Unidades",
        "ubicacion_almacen": "Bodega Central - Galpón B",
        "distribuidor_principal": "Cerámicas del Pacífico",
        "precio_unitario_clp": 420,
        "precio_al_por_mayor_clp": 360,
        "minimo_por_mayor": 1000,
        "estado_stock": "Disponible"
    }
]
pd.DataFrame(inventario_data).to_csv("constructora_data/inventario_materiales.csv", index=False, encoding="utf-8")

# 2. Generar CSV Nómina y Personal
personal_data = [
    {
        "id_trabajador": "EMP-101",
        "nombre_completo": "Carlos Mendoza Tapia",
        "rol_posicion": "Jefe de Obra Senior",
        "tipo_contrato": "Indefinido",
        "especialidad": "Estructuras y Obra Gruesa",
        "proyecto_actual": "Torre Miramar Valparaíso",
        "proyectos_anteriores": "Edificio Costa Sol, Pavimentación Av. España",
        "certificaciones": "Prevención de Riesgos OIT, Licencia Grúa Horquilla",
        "estado_laboral": "Activo",
        "evaluacion_desempeno": "Sobresaliente",
        "posibilidad_ascenso": "Sí (Candidato a Director de Operaciones)"
    },
    {
        "id_trabajador": "EMP-102",
        "nombre_completo": "María Paz Salamanca",
        "rol_posicion": "Ingeniera Civil Estructural",
        "tipo_contrato": "Indefinido",
        "especialidad": "Cálculo Estructural y BIM",
        "proyecto_actual": "Condominio EcoHabitat Viña",
        "proyectos_anteriores": "Paso Superior Marga Marga",
        "certificaciones": "Revit BIM Professional, Certificación LEED Green Associate",
        "estado_laboral": "Activo",
        "evaluacion_desempeno": "Excelente",
        "posibilidad_ascenso": "Sí (Candidata a Jefa de Proyecto)"
    },
    {
        "id_trabajador": "SUB-201",
        "nombre_completo": "Roberto Gómez Silva (Electricidad Silva EIRL)",
        "rol_posicion": "Subcontratista Electricista",
        "tipo_contrato": "Por Obra / Servicios",
        "especialidad": "Instalaciones Eléctricas de Alta y Baja Tensión",
        "proyecto_actual": "Parque Logístico Curauma",
        "proyectos_anteriores": "Torre Miramar Valparaíso, Colegio San Pedro",
        "certificaciones": "Certificación SEC Clase A",
        "estado_laboral": "En Proyecto",
        "evaluacion_desempeno": "Bueno",
        "posibilidad_ascenso": "N/A (Subcontratista Preferente)"
    }
]
pd.DataFrame(personal_data).to_csv("constructora_data/nomina_y_personal.csv", index=False, encoding="utf-8")

# 3. Generar PDF Catálogo de Proyectos
html_content = """
<!DOCTYPE html>
<html>
<head><style>body { font-family: Arial; padding: 20px; } h1 { color: #0284c7; }</style></head>
<body>
  <h1>Constructora Nova Build SpA</h1>
  <h2>Catálogo de Servicios y Proyectos</h2>
  <h3>1. Especialidades</h3>
  <p>Edificación Residencial en Altura, Condominios Sustentables con certificación LEED, e Infraestructura Logística.</p>
  <h3>2. Proyectos En Desarrollo</h3>
  <p><strong>Torre Miramar Valparaíso:</strong> 65% de avance. Edificio residencial de 18 pisos. Testimonio: 'Excelente avance y resistencia sísmica.' — Inmobiliaria del Mar.</p>
  <p><strong>Condominio EcoHabitat Viña:</strong> 30% de avance. 45 casas sostenibles en Reñaca.</p>
  <p><strong>Parque Logístico Curauma:</strong> 85% de avance. Centro de distribución de 12.000 m2.</p>
</body>
</html>
"""
HTML(string=html_content).write_pdf("constructora_data/catalogo_y_proyectos.pdf")
print("✅ Archivos creados exitosamente en 'constructora_data/'")

## Paso 3: Configuración de la API Key de Google Gemini
Introduce tu API Key de Gemini a continuación para conectar el LLM y los embeddings.

In [ ]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ or not os.environ["GOOGLE_API_KEY"]:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Introduce tu GOOGLE_API_KEY de Gemini: ")

## Paso 4: Definición de Herramientas y Agente Multi-Herramienta (LangChain)

In [ ]:
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.tools import Tool
from langchain.agents import initialize_agent, AgentType

# Inicializar LLM
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.2)

# --- HERRAMIENTA 1: RAG PDF (Proyectos) ---
loader = PyPDFLoader("constructora_data/catalogo_y_proyectos.pdf")
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs_split = text_splitter.split_documents(docs)

embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
vectorstore = Chroma.from_documents(docs_split, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

def consultar_catalogo(query: str) -> str:
    results = retriever.get_relevant_documents(query)
    context = "\n---\n".join([doc.page_content for doc in results])
    prompt = f"Usa la información de la constructora Nova Build para responder:\n\nContexto:\n{context}\n\nPregunta: {query}"
    return llm.invoke(prompt).content

# --- HERRAMIENTA 2: CSV Inventario ---
df_inventario = pd.read_csv("constructora_data/inventario_materiales.csv")
def consultar_inventario(query: str) -> str:
    context_str = df_inventario.to_string(index=False)
    prompt = f"Eres el asistente de inventario de Nova Build. Responde basándote en esta tabla:\n\n{context_str}\n\nConsulta: {query}"
    return llm.invoke(prompt).content

# --- HERRAMIENTA 3: CSV Nómina ---
df_nomina = pd.read_csv("constructora_data/nomina_y_personal.csv")
def consultar_nomina(query: str) -> str:
    context_str = df_nomina.to_string(index=False)
    prompt = f"Eres el asistente de RRHH de Nova Build. Responde basándote en esta lista de personal:\n\n{context_str}\n\nConsulta: {query}"
    return llm.invoke(prompt).content

# Registrar herramientas
tools = [
    Tool(
        name="Catalogo_Proyectos_PDF",
        func=consultar_catalogo,
        description="Util para consultar proyectos en desarrollo, avances de obra, calidad y testimonios de clientes."
    ),
    Tool(
        name="Gestion_Inventario_CSV",
        func=consultar_inventario,
        description="Util para consultar stock de materiales, proveedores, precios unitarios y al por mayor en bodega."
    ),
    Tool(
        name="Nomina_Personal_CSV",
        func=consultar_nomina,
        description="Util para consultar trabajadores, subcontratistas, cargos, certificaciones y posibilidad de ascenso."
    )
]

# Inicializar Agente ReAct
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

print("🤖 ¡Agente BuildTech listo para consultas!")

## Paso 5: Pruebas del Agente en Vivo

In [ ]:
# Prueba 1: Captación / RAG PDF
print("--- PRUEBA 1 ---")
print(agent.run("¿Qué porcentaje de avance tiene la Torre Miramar y qué opinan sus clientes?"))

# Prueba 2: Inventario CSV
print("\n--- PRUEBA 2 ---")
print(agent.run("¿Cuál es el distribuidor del Cemento Melón y qué precio tiene al por mayor?"))

# Prueba 3: Nómina CSV
print("\n--- PRUEBA 3 ---")
print(agent.run("¿Qué trabajadora cuenta con certificación LEED y qué opción de ascenso tiene?"))